In [1]:
import numpy as np

In [2]:
# Parameters
real_noise_std = 1e-10
noise_assumption = 1e-10
rbf_lengthscale = 0.1
beta = 1.96  # Exploration-exploitation trade-off

# Generate random multi-modal function
modes = np.random.randint(1, 5)
std = np.random.uniform(low=0.005, high=0.05, size=modes)
means = np.random.uniform(size=modes)
amps = np.random.uniform(size=modes)

# Define black-box function (sum of Gaussians)
def calc_function(x):
    exp = -(x - means)**2 / std
    y = amps * np.exp(exp)
    return np.sum(y)

In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')
model = GaussianProcessRegressor(kernel=kernel, alpha=noise_assumption)

X, Y = [], []  # Query points and observations
max_obs = 0  # Best observation so far
x_grid = np.linspace(0, 1, 101).reshape(-1, 1)  # Evaluation grid
num_queries = 5  # Optimization budget

In [8]:
from matplotlib import pyplot as plt
from IPython.core.display_functions import clear_output

for i in range(num_queries):
  clear_output(wait=True)  # Refresh display

  # Fit GP once we have at least one observation
  if len(X) > 0:
    model.fit(np.array(X).reshape(-1, 1), np.array(Y).ravel())
    post_mean, post_std = model.predict(x_grid, return_std=True)
  else:
    # Prior mean and uncertainty before the first observation
    post_mean = np.zeros(x_grid.shape[0])
    post_std = np.sqrt(noise_assumption) * np.ones(x_grid.shape[0])

  # Visualization
  plt.figure(figsize=(15,7))
  plt.scatter(X, Y, c='r')  # Observations
  plt.plot(x_grid.squeeze(), post_mean)
  plt.fill_between(
    x_grid.squeeze(),
    post_mean - beta*post_std,
    post_mean + beta*post_std,
    alpha = 0.2,
    label = str(beta) + ' Standard Deviations'
   )


  # UCB acquisition function
  acquisition_function = post_mean + beta * post_std

  # Point selection
  if i == 0:
     x = np.random.uniform(0,1)  # Initial random point
  else:
      x = x_grid[np.argmax(acquisition_function)].item()  # Maximise UCB

  # Evaluate function
  y = calc_function(x) + noise_assumption
  X.append(x)
  Y.append(y)

  # Track best observation
  max_obs = max(max_obs, y)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

In [ ]:
# Evaluate true function on dense grid
x_dense = np.linspace(0, 1, 1001)
y_real = [calc_function(x) for x in x_dense]
best_obs_grid = max(y_real)  # True maximum

# Final GP fit
model.fit(X, Y)
post_mean, post_std = model.predict(x_dense, return_std=True)

In [ ]:
# Plot results
plt.figure(figsize=(15,7))
plt.plot(x_dense, y_real, 'k', label='True function')
plt.scatter(X, Y, c='r', label='Queries')
plt.plot(x_dense, post_mean, label='GP mean')
plt.fill_between(
    x_grid.squeeze(),
    post_mean - beta*post_std,
    post_mean + beta*post_std,
    alpha = 0.2,
    label = str(beta) + ' Standard Deviations'
)

print(f"True maximum: {best_obs_grid:.4f}")
print(f"Best found: {max_obs:.4f}")

In [12]:
acquisition_function = post_std ** 2  # For variance

# or

acquisition_function = post_std  # For standard deviation

In [13]:
from scipy.stats import norm

if i == 0:
    acquisition_function = np.ones_like(post_mean)  # all probabilities are equal for the first point

else:
   y_max = np.max(Y)
   z = (post_mean - y_max) / (post_std + 1e-12)
   acquisition_function = norm.cdf(z)

In [14]:
eta = 0.01 # encourages improvement by at least eta
y_max = float(np.max(Y))
z = (post_mean - y_max - eta) / (post_std + 1e-12)
acquisition_function = norm.cdf(z)